## Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

## Data Preparation

### Load Data

In [ ]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
Y_train = train['label']
X_train = train.drop(labels=['label'], axis=1)

del train

In [ ]:
Y_train.value_counts()

### Normalization

In [ ]:
X_train = X_train / 255.0
X_test = test / 255.0

### Reshape

In [ ]:
X_train = X_train.values.reshape(-1, 28, 28, 1)
X_test = X_test.values.reshape(-1, 28, 28, 1)

### Label Encoding

In [ ]:
Y_train = to_categorical(Y_train, num_classes=10)

### Split data

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.1, random_state=42)

### Preview Data

In [ ]:
plt.figure(figsize=(20, 10))

for i in range(50):
    plt.subplot(5, 10, i+1)
    plt.imshow(X_train[i].reshape((28,28)), cmap=plt.cm.binary)
    plt.title(np.argmax(Y_train[i]), fontsize=25)
    plt.axis('off')

plt.subplots_adjust(hspace=0.75)
plt.show()

## Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train)

In [ ]:
for k in range(5):
    sample_image = X_train[k].reshape((1, 28, 28, 1))
    augmented_images = datagen.flow(sample_image, batch_size=1)
    
    plt.figure(figsize=(15, 5))
    
    for i in range(10):
        augmented_image = next(augmented_images)[0]
        plt.subplot(1, 10, i + 1)
        plt.imshow(augmented_image.reshape(28, 28), cmap=plt.cm.binary)
        plt.axis('off')

    plt.show()

## Model

### Define the Model

In [ ]:
model = Sequential()

model.add(Conv2D(filters = 32, kernel_size = (5, 5), padding = 'Same', activation ='relu', input_shape = (28, 28, 1)))
model.add(Conv2D(filters = 32, kernel_size = (5, 5), padding = 'Same', activation ='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(filters = 64, kernel_size = (3, 3), padding = 'Same', activation ='relu'))
model.add(Conv2D(filters = 64, kernel_size = (3, 3), padding = 'Same', activation ='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(256, activation = "relu"))
model.add(Dropout(0.5))
model.add(Dense(10, activation = "softmax"))

### Compile the Model

In [ ]:
model.compile(optimizer="adam", loss = "categorical_crossentropy", metrics=["accuracy"])

### Fit the Model

In [ ]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_accuracy', patience=3, verbose=1, factor=0.5, min_lr=0.00001)

In [ ]:
history = model.fit(datagen.flow(X_train, Y_train, batch_size=64), epochs=30, validation_data=(X_val, Y_val), verbose = 2, steps_per_epoch=X_train.shape[0] // 64, callbacks=[learning_rate_reduction])

## Evaluate the Model

### Training and validation curves

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.show()

### Predict

In [ ]:
X_test = test.values.reshape(-1, 28, 28, 1)

results = model.predict(X_test)
results = np.argmax(results, axis=1)
results = pd.Series(results, name="Label")

In [ ]:
submission = pd.concat([pd.Series(range(1, 28001), name = "ImageId"), results], axis = 1)
submission.to_csv("cnn_result.csv", index=False)

### Save the Model

In [ ]:
model.save('../models/digit_cnn.keras')